# آزمون اختیاری دریافت واقعی

پس از موفقیت دفترچه‌های آفلاین، RUN_LIVE را True کن. این دفترچه فقط دریافت داده را می‌آزماید؛ تحلیل و هوش مصنوعی ندارد.

### سلول ۱ — ورود کتابخانه‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import pydara
from pydara import tse
from pydara.tse import Client, FetchPolicy
print(pydara.__version__)

### سلول ۲ — تابع گزارش خروجی

In [ ]:
def show_result(frame):
    display(tse.select_columns(frame, "compact"))
    print(tse.frame_metadata(frame))

### سلول ۳ — تنظیمات

In [ ]:
RUN_LIVE = False
SYMBOL = "فولاد"
START = "1403-01-01"
END = "1403-12-30"
OUTPUT = Path("dara_live_review_output")

### سلول ۴ — اجرا

In [ ]:
if RUN_LIVE:
    with Client() as client:
        quote_frame = client.get_quote_frame(SYMBOL, policy=FetchPolicy.fresh())
        history = client.get_history(SYMBOL, start=START, end=END, calendar="jalali", include_jdate=True)
        print(client.storage_status())
    show_result(quote_frame)
    show_result(history)
    display(tse.validate(history, schema="daily_bars", strict=False))
else:
    print("دریافت زنده انجام نشد. برای اجرای صریح RUN_LIVE را True کن.")

### سلول ۵ — خروجی صریح فایل

In [ ]:
if RUN_LIVE:
    destination = OUTPUT / "history.csv"
    tse.save(history, destination, overwrite=True)
    restored = tse.load(destination)
    pd.testing.assert_frame_equal(history, restored, check_dtype=True)
    print("CSV roundtrip OK:", destination.resolve())